In [1]:
# Cell 1: Imports + config
from pathlib import Path
import json
import gc

import pandas as pd
import pyarrow.parquet as pq

EVENT_COLS = [
    "match_id",
    "minute",
    "type.name",
    "player.id",
    "player.name",
    "team.name",
    "location",
    "pass.length",
    "shot.statsbomb_xg",
    "dribble.outcome.name",
]

WINGER_POSITIONS = {
    "Left Wing",
    "Right Wing",
    "Left Midfield",
    "Right Midfield",
}

events_path = Path("../data/processed/events_light.parquet")
wingers_path = Path("../data/processed/wingers.parquet")
lineups_path = Path("../data/raw/statsbomb/lineups")
output_path = Path("../data/processed/winger_events.parquet")

events_path, wingers_path, lineups_path, output_path


(WindowsPath('../data/processed/events_light.parquet'),
 WindowsPath('../data/processed/wingers.parquet'),
 WindowsPath('../data/raw/statsbomb/lineups'),
 WindowsPath('../data/processed/winger_events.parquet'))

In [2]:
# Cell 2: Helpers
def normalize_wingers(df: pd.DataFrame) -> pd.DataFrame:
    required = ["match_id", "player_id", "start_minute", "end_minute"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"wingers dataframe missing columns: {missing}")

    out = df.copy()

    # Keep optional columns if they exist
    keep_cols = [
        "match_id", "player_id", "player_name", "team_name",
        "position", "start_minute", "end_minute"
    ]
    keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[keep_cols]

    out["match_id"] = pd.to_numeric(out["match_id"], errors="coerce").astype("Int64")
    out["player_id"] = pd.to_numeric(out["player_id"], errors="coerce").astype("Int64")
    out["start_minute"] = pd.to_numeric(out["start_minute"], errors="coerce")
    out["end_minute"] = pd.to_numeric(out["end_minute"], errors="coerce")

    # If "to" is missing, keep events until match end incl. extra-time
    out["start_minute"] = out["start_minute"].fillna(0)
    out["end_minute"] = out["end_minute"].fillna(130)

    out = out.dropna(subset=["match_id", "player_id"])
    out = out[out["end_minute"] >= out["start_minute"]]
    out = out.reset_index(drop=True)
    return out


def build_wingers_from_lineups(lineups_dir: Path) -> pd.DataFrame:
    rows = []
    for file in lineups_dir.glob("*.json"):
        match_id = int(file.stem)
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        for team in data:
            team_name = team.get("team_name")
            for player in team.get("lineup", []):
                for pos in player.get("positions", []):
                    position_name = pos.get("position")
                    if position_name in WINGER_POSITIONS:
                        rows.append({
                            "match_id": match_id,
                            "player_id": player.get("player_id"),
                            "player_name": player.get("player_name"),
                            "team_name": team_name,
                            "position": position_name,
                            "start_minute": pos.get("from"),
                            "end_minute": pos.get("to"),
                        })

    return pd.DataFrame(rows)


In [3]:
# Cell 3: Load wingers (preferred: processed file; fallback: lineups)
if wingers_path.exists():
    df_wingers = pd.read_parquet(wingers_path)
else:
    df_wingers = build_wingers_from_lineups(lineups_path)

df_wingers = normalize_wingers(df_wingers)

print("Wingers rows:", len(df_wingers))
print("Unique matches:", df_wingers["match_id"].nunique())
print("Columns:", list(df_wingers.columns))
df_wingers.head()


Wingers rows: 27297
Unique matches: 3374
Columns: ['match_id', 'player_id', 'player_name', 'team_name', 'position', 'start_minute', 'end_minute']


,match_id,player_id,player_name,team_name,position,start_minute,end_minute
0,15946,3501,Philippe Coutinho Correia,Barcelona,Left Wing,0.0,130.0
1,15946,5477,Ousmane Dembélé,Barcelona,Left Midfield,0.0,130.0
2,15946,5477,Ousmane Dembélé,Barcelona,Left Wing,0.0,130.0
3,15946,5503,Lionel Andrés Messi Cuccittini,Barcelona,Right Wing,0.0,130.0
4,15946,6379,Sergi Roberto Carnicer,Barcelona,Right Midfield,0.0,130.0


In [4]:
# Cell 4: Chunked extraction from events parquet (row-group based)
rename_map = {
    "player.id": "player_id",
    "player.name": "player_name",
    "team.name": "team_name",
}

winger_matches = set(df_wingers["match_id"].dropna().astype(int).tolist())
winger_players = set(df_wingers["player_id"].dropna().astype(int).tolist())

pf = pq.ParquetFile(events_path)
parts = []

print("Row groups:", pf.num_row_groups)

for rg_idx in range(pf.num_row_groups):
    chunk = pf.read_row_group(rg_idx, columns=EVENT_COLS).to_pandas()
    chunk = chunk.rename(columns=rename_map)

    chunk["match_id"] = pd.to_numeric(chunk["match_id"], errors="coerce").astype("Int64")
    chunk["player_id"] = pd.to_numeric(chunk["player_id"], errors="coerce").astype("Int64")
    chunk["minute"] = pd.to_numeric(chunk["minute"], errors="coerce")

    # Early reduction before merge
    chunk = chunk[chunk["match_id"].isin(winger_matches)]
    chunk = chunk[chunk["player_id"].isin(winger_players)]

    if chunk.empty:
        continue

    merged = chunk.merge(
        df_wingers,
        on=["match_id", "player_id"],
        how="inner",
        suffixes=("", "_winger"),
    )

    merged = merged[
        (merged["minute"] >= merged["start_minute"]) &
        (merged["minute"] <= merged["end_minute"])
    ]

    if not merged.empty:
        parts.append(merged)

    if rg_idx % 10 == 0:
        gc.collect()

if parts:
    df_winger_events = pd.concat(parts, ignore_index=True)
else:
    # Keep a stable schema even if empty
    df_winger_events = pd.DataFrame(columns=[
        "match_id", "minute", "type.name", "player_id", "player_name", "team_name",
        "location", "pass.length", "shot.statsbomb_xg", "dribble.outcome.name",
        "player_name_winger", "team_name_winger", "position", "start_minute", "end_minute"
    ])

print("Winger events rows:", len(df_winger_events))
df_winger_events.head()


Row groups: 10
Winger events rows: 2656669


,match_id,minute,type.name,player_id,player_name,team_name,location,pass.length,shot.statsbomb_xg,dribble.outcome.name,player_name_winger,team_name_winger,position,start_minute,end_minute
0,15946,0,Pass,6581,Jonathan Rodríguez Menéndez,Deportivo Alavés,"[61.0, 40.1]",29.76995,NaN,None,Jonathan Rodríguez Menéndez,Deportivo Alavés,Left Midfield,0.0,130.0
1,15946,0,Ball Receipt*,5477,Ousmane Dembélé,Barcelona,"[36.2, 5.3]",NaN,NaN,None,Ousmane Dembélé,Barcelona,Left Midfield,0.0,130.0
2,15946,0,Ball Receipt*,5477,Ousmane Dembélé,Barcelona,"[36.2, 5.3]",NaN,NaN,None,Ousmane Dembélé,Barcelona,Left Wing,0.0,130.0
3,15946,0,Carry,5477,Ousmane Dembélé,Barcelona,"[36.2, 5.3]",NaN,NaN,None,Ousmane Dembélé,Barcelona,Left Midfield,0.0,130.0
4,15946,0,Carry,5477,Ousmane Dembélé,Barcelona,"[36.2, 5.3]",NaN,NaN,None,Ousmane Dembélé,Barcelona,Left Wing,0.0,130.0


In [5]:
# Cell 5: Save
output_path.parent.mkdir(parents=True, exist_ok=True)
df_winger_events.to_parquet(output_path, index=False)
print(f"Saved: {output_path.resolve()}")


Saved: C:\Users\ismail\football-scouting-platform\data\processed\winger_events.parquet


In [6]:
# Cell 6: Sanity checks
print("Shape:", df_winger_events.shape)
print("Unique players:", df_winger_events["player_id"].nunique() if len(df_winger_events) else 0)
print("Unique matches:", df_winger_events["match_id"].nunique() if len(df_winger_events) else 0)

if len(df_winger_events):
    bad = df_winger_events[
        (df_winger_events["minute"] < df_winger_events["start_minute"]) |
        (df_winger_events["minute"] > df_winger_events["end_minute"])
    ]
    print("Out-of-window rows:", len(bad))


Shape: (2656669, 15)
Unique players: 3226
Unique matches: 2878
Out-of-window rows: 0


In [7]:
df_winger_events["type.name"].value_counts().head(10)

type.name
Ball Receipt*    766106
Pass             587848
Carry            572222
Pressure         289767
Ball Recovery     81819
Duel              58290
Dribble           51499
Miscontrol        39874
Dispossessed      35198
Shot              31676
Name: count, dtype: int64